# Couplings as transport polylines

This notebook generates `fig:kantorovich-coupling-polylines`.  For two discrete measures
\[
\alpha=\sum_i a_i\delta_{x_i},\qquad
\beta=\sum_j b_j\delta_{y_j},
\]
a coupling is a nonnegative matrix `P` with prescribed row and column sums.  The figure uses the canonical red/blue clouds from the matching section and compares three couplings: a deterministic graph, the independent product plan, and the optimal Kantorovich plan for the quadratic cost.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from scipy.optimize import linear_sum_assignment

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    RED,
    VIOLET,
    box_axes,
    canonical_matching_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
np.random.seed(0)

POINT_SIZE = 10.5
POINT_SIZE_LARGE = 12.0


def active_pairs(P, threshold=1e-11):
    return [(i, j, float(P[i, j])) for i in range(P.shape[0]) for j in range(P.shape[1]) if P[i, j] > threshold]


def mass_sizes(weights, base=POINT_SIZE_LARGE):
    weights = np.asarray(weights, dtype=float)
    weights = weights / max(weights.max(), 1e-15)
    return base * (0.58 + 1.25 * np.sqrt(weights))


def draw_clouds(ax, x, y, a=None, b=None, *, alpha=1.0, size=POINT_SIZE, zorder=4):
    sx = np.full(len(x), size) if a is None else mass_sizes(a, base=size)
    sy = np.full(len(y), size) if b is None else mass_sizes(b, base=size)
    ax.scatter(x[:, 0], x[:, 1], s=sx, marker="o", color=RED, alpha=alpha, edgecolor="none", zorder=zorder)
    ax.scatter(y[:, 0], y[:, 1], s=sy, marker="o", color=BLUE, alpha=alpha, edgecolor="none", zorder=zorder)


def set_cloud_limits(ax, x, y, pad=0.18):
    (xmin, xmax), (ymin, ymax) = padded_limits(np.vstack([x, y]), pad=pad)
    span = max(xmax - xmin, ymax - ymin)
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    ax.set_xlim(cx - span / 2, cx + span / 2)
    ax.set_ylim(cy - span / 2, cy + span / 2)
    ax.set_aspect("equal")
    remove_axes(ax)


def draw_straight_segments(ax, x, y, pairs, *, color=VIOLET, min_width=0.12, max_width=1.20, alpha=0.55, zorder=1):
    if not pairs:
        return
    masses = np.array([m for _, _, m in pairs], dtype=float)
    rel = masses / max(masses.max(), 1e-15)
    segments = [[x[i], y[j]] for i, j, _ in pairs]
    base = np.array(to_rgb(color))
    colors = [(*base, min(alpha * (0.22 + 0.78 * np.sqrt(r)), 0.9)) for r in rel]
    widths = min_width + (max_width - min_width) * np.sqrt(rel)
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=widths, zorder=zorder))


def draw_curved_segments(ax, x, y, pairs, *, color=VIOLET, min_width=0.10, max_width=1.20, alpha=0.55, curvature=0.11, zorder=1):
    if not pairs:
        return
    masses = np.array([m for _, _, m in pairs], dtype=float)
    rel = masses / max(masses.max(), 1e-15)
    base = np.array(to_rgb(color))
    curves, colors, widths = [], [], []
    for (i, j, _), r in zip(pairs, rel):
        p, q = x[i], y[j]
        mid = 0.5 * (p + q)
        direction = q - p
        normal = np.array([-direction[1], direction[0]])
        norm = np.linalg.norm(normal)
        if norm > 1e-12:
            normal = normal / norm
        sign = -1 if ((i + 2 * j) % 2) else 1
        control = mid + sign * curvature * normal * np.linalg.norm(direction)
        ts = np.linspace(0, 1, 16)
        curve = ((1 - ts)[:, None] ** 2) * p + (2 * (1 - ts) * ts)[:, None] * control + (ts[:, None] ** 2) * q
        curves.append(curve)
        colors.append((*base, min(alpha * (0.22 + 0.78 * np.sqrt(r)), 0.88)))
        widths.append(min_width + (max_width - min_width) * np.sqrt(r))
    ax.add_collection(LineCollection(curves, colors=colors, linewidths=widths, zorder=zorder))

## Shared geometry

The source is a compact central Gaussian cloud and the target is the same three-component cloud used in `fig:matching-2d-cost-exponent`.  Keeping the geometry fixed makes the difference between couplings visible as a difference in edge structure, not as a change of data.

In [ ]:
fig_name = "kantorovich-coupling-polylines"
out = figure_dir(fig_name)

x, y, labels = canonical_matching_clouds(seed=2027, n_source=24)
n = len(x)
a = np.ones(n) / n
b = np.ones(n) / n
C = ot.dist(x, y, metric="euclidean") ** 2

# A deliberately non-optimal deterministic graph, obtained by rotating the
# optimal assignment.  It is still a valid Monge-type coupling for uniform masses.
rows, cols = linear_sum_assignment(C)
opt_perm = np.empty(n, dtype=int)
opt_perm[rows] = cols
graph_perm = np.roll(opt_perm, 5)

P_graph = np.zeros((n, n))
P_graph[np.arange(n), graph_perm] = 1 / n
P_product = np.outer(a, b)
P_opt = ot.emd(a, b, C, numItermax=200000)

## Exported panels

Each positive coefficient `P_ij` is drawn as a red-to-blue polyline.  Width and opacity encode transported mass; the product plan is therefore dense and faint, while deterministic plans have one edge per source point.

In [ ]:
panels = [
    ("graph.pdf", P_graph, dict(min_width=0.26, max_width=0.55, alpha=0.58, curvature=0.10)),
    ("product.pdf", P_product, dict(min_width=0.045, max_width=0.22, alpha=0.25, curvature=0.13)),
    ("optimal.pdf", P_opt, dict(min_width=0.26, max_width=0.62, alpha=0.62, curvature=0.08)),
]

for filename, P, opts in panels:
    fig, ax = plt.subplots(figsize=(2.15, 2.15))
    draw_curved_segments(ax, x, y, active_pairs(P), color=VIOLET, zorder=1, **opts)
    draw_clouds(ax, x, y, size=POINT_SIZE, zorder=3)
    set_cloud_limits(ax, x, y, pad=0.18)
    save_pdf(fig, out / filename)
    plt.close(fig)